In [0]:
"""
AeroPulse Enterprise Lakehouse Platform.

Ingestion registry utilities.

This module manages source delivery ingestion status
for rerun-safe Bronze processing.
"""


def is_delivery_processed(
    spark,
    registry_table: str,
    source_delivery_path: str,
) -> bool:
    """
    Check whether a source delivery has already been
    processed successfully.
    """

    escaped_path = source_delivery_path.replace(
        "'",
        "''"
    )

    result_df = spark.sql(f"""
        SELECT COUNT(*) AS processed_count
        FROM {registry_table}
        WHERE source_file_path = '{escaped_path}'
          AND ingestion_status = 'SUCCESS'
    """)

    processed_count = (
        result_df
        .collect()[0]["processed_count"]
    )

    return processed_count > 0


def register_delivery_start(
    spark,
    registry_table: str,
    source_delivery_path: str,
    source_file_name: str,
    source_system: str,
    source_entity: str,
    file_format: str,
    pipeline_run_id: str,
) -> None:
    """
    Register a source delivery as RUNNING.
    """

    spark.sql(f"""
        INSERT INTO {registry_table}
        VALUES
        (
            '{source_delivery_path}',
            '{source_file_name}',
            '{source_system}',
            '{source_entity}',
            '{file_format}',

            '{pipeline_run_id}',
            'RUNNING',

            NULL,
            NULL,

            current_timestamp(),
            NULL,

            NULL,

            current_timestamp(),
            current_timestamp()
        )
    """)


def mark_delivery_success(
    spark,
    registry_table: str,
    source_delivery_path: str,
    pipeline_run_id: str,
    records_read: int,
    records_inserted: int,
) -> None:
    """
    Mark a source delivery as successfully processed.
    """

    escaped_path = source_delivery_path.replace(
        "'",
        "''"
    )

    spark.sql(f"""
        UPDATE {registry_table}
        SET
            ingestion_status = 'SUCCESS',
            records_read = {records_read},
            records_inserted = {records_inserted},
            ingestion_end_timestamp = current_timestamp(),
            updated_timestamp = current_timestamp()
        WHERE source_file_path = '{escaped_path}'
          AND pipeline_run_id = '{pipeline_run_id}'
          AND ingestion_status = 'RUNNING'
    """)


def mark_delivery_failed(
    spark,
    registry_table: str,
    source_delivery_path: str,
    pipeline_run_id: str,
    error_message: str,
) -> None:
    """
    Mark a source delivery as FAILED.
    """

    escaped_path = source_delivery_path.replace(
        "'",
        "''"
    )

    escaped_error_message = error_message.replace(
        "'",
        "''"
    )

    spark.sql(f"""
        UPDATE {registry_table}
        SET
            ingestion_status = 'FAILED',
            error_message = '{escaped_error_message}',
            ingestion_end_timestamp = current_timestamp(),
            updated_timestamp = current_timestamp()
        WHERE source_file_path = '{escaped_path}'
          AND pipeline_run_id = '{pipeline_run_id}'
          AND ingestion_status = 'RUNNING'
    """)